# Customer Subscription Prediction & Marketing Analysis
**Author:** Darius Nobles  
**Date:** 2025  
**Tools:** Python, Pandas, Scikit-learn, Matplotlib, Seaborn, SQL  
**Dataset:** Bank Marketing Dataset (UCI Machine Learning Repository) — 45,211 records

---

## Business Problem

A subscription-based financial services company runs outbound telemarketing campaigns to sell term deposit subscriptions. Each call costs the company time and money. The marketing team needs to know:

> **Which customers are most likely to subscribe — so we can prioritize outreach and reduce wasted spend?**

**Goal:** Build a machine learning model to predict customer subscription likelihood, and surface actionable insights that the marketing team can act on immediately.

---

## Table of Contents
1. [Data Loading & First Look](#1)
2. [Data Cleaning & Preprocessing](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Feature Engineering](#4)
5. [Model Building & Evaluation](#5)
6. [Key Findings & Business Recommendations](#6)

---
## Section 1: Data Loading & First Look <a id='1'></a>

In [ ]:
# ── Import all libraries needed for this project ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

# ── Set a consistent visual style for all charts ──
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Custom color palette aligned with the project brand
PALETTE = ['#2C6FBF', '#E05C2A', '#2CA87F', '#F5C842', '#7B5EA7']
sns.set_palette(PALETTE)

print('Libraries loaded successfully.')

In [ ]:
# ── Load the dataset ──
# Source: UCI Bank Marketing Dataset
# Download from: https://archive.ics.uci.edu/ml/datasets/Bank+Marketing
# File: bank-additional-full.csv  (use semicolon separator)

df = pd.read_csv('data/bank-additional-full.csv', sep=';')

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── Get a quick overview of data types and missing values ──
print('=== DATA TYPES ===')
print(df.dtypes)
print()
print('=== MISSING VALUES ===')
print(df.isnull().sum())
print()
print('=== TARGET VARIABLE DISTRIBUTION ===')
print(df['y'].value_counts())
print(f"\nSubscription rate: {df['y'].value_counts(normalize=True)['yes']*100:.1f}%")

In [ ]:
# ── Statistical summary of numerical columns ──
df.describe().round(2)

---
## Section 2: Data Cleaning & Preprocessing <a id='2'></a>

In [ ]:
# ── Handle 'unknown' values — treated as a category, not null ──
# The dataset uses 'unknown' as a placeholder for missing categorical data
unknown_counts = (df == 'unknown').sum()
print('Columns with unknown values:')
print(unknown_counts[unknown_counts > 0])

In [ ]:
# ── Remove the 'duration' column ──
# Duration (call length) is only known AFTER the call happens.
# Including it would cause data leakage — the model would "cheat".
# In a real deployment, we wouldn't know duration before making the call.
df = df.drop(columns=['duration'])

# ── Encode the target variable: yes → 1, no → 0 ──
df['subscribed'] = (df['y'] == 'yes').astype(int)
df = df.drop(columns=['y'])

print('Target encoded. Sample:')
print(df['subscribed'].value_counts())

In [ ]:
# ── One-hot encode all categorical columns ──
# This converts text categories into numbers so our model can use them
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Encoding {len(categorical_cols)} categorical columns: {categorical_cols}')

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(f'\nShape after encoding: {df_encoded.shape}')

---
## Section 3: Exploratory Data Analysis (EDA) <a id='3'></a>

> Before modeling, we need to understand what's in the data. Good EDA surfaces insights that shape both the model and the business recommendations.

In [ ]:
# ── Chart 1: Subscription Rate by Job Type ──
# Business question: Which customer segments have the highest subscription rates?

job_sub = df.groupby('job')['subscribed'].mean().sort_values(ascending=True) * 100

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(job_sub.index, job_sub.values,
               color=['#2C6FBF' if v >= job_sub.mean() else '#B0C4DE' for v in job_sub.values],
               edgecolor='none', height=0.6)

ax.axvline(job_sub.mean(), color='#E05C2A', linestyle='--', linewidth=1.2,
           label=f'Average: {job_sub.mean():.1f}%')

for bar, val in zip(bars, job_sub.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9, color='#333')

ax.set_xlabel('Subscription Rate (%)', fontsize=11)
ax.set_title('Subscription Rate by Job Type', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('images/01_subscription_by_job.png', bbox_inches='tight')
plt.show()
print('Chart saved to images/01_subscription_by_job.png')

In [ ]:
# ── Chart 2: Subscription Rate by Month ──
# Business question: Are there seasonal patterns in subscription behavior?

month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
month_sub = df.groupby('month')['subscribed'].mean().reindex(month_order) * 100

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(month_sub.index, month_sub.values,
        color='#2C6FBF', linewidth=2.5, marker='o', markersize=8, markerfacecolor='white',
        markeredgewidth=2)
ax.fill_between(month_sub.index, month_sub.values, alpha=0.12, color='#2C6FBF')

for i, (month, val) in enumerate(month_sub.items()):
    ax.annotate(f'{val:.1f}%', (month, val),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=8.5, color='#555')

ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Subscription Rate (%)', fontsize=11)
ax.set_title('Subscription Rate by Month — Seasonal Patterns', fontsize=14, fontweight='bold', pad=15)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('images/02_subscription_by_month.png', bbox_inches='tight')
plt.show()
print('Chart saved to images/02_subscription_by_month.png')

In [ ]:
# ── Chart 3: Age Distribution — Subscribers vs Non-Subscribers ──
# Business question: Does age predict subscription likelihood?

fig, ax = plt.subplots(figsize=(11, 5))

df[df['subscribed'] == 1]['age'].plot(kind='hist', bins=30, alpha=0.65,
                                       color='#2C6FBF', label='Subscribed', ax=ax)
df[df['subscribed'] == 0]['age'].plot(kind='hist', bins=30, alpha=0.65,
                                       color='#E05C2A', label='Did not subscribe', ax=ax)

ax.set_xlabel('Age', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Age Distribution: Subscribers vs Non-Subscribers', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('images/03_age_distribution.png', bbox_inches='tight')
plt.show()
print('Chart saved to images/03_age_distribution.png')

In [ ]:
# ── Chart 4: Number of Contacts vs Subscription Rate ──
# Business question: Does calling customers more times help or hurt conversion?

contact_sub = df.groupby('campaign')['subscribed'].mean() * 100
contact_sub = contact_sub[contact_sub.index <= 15]  # Cap at 15 for readability

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(contact_sub.index, contact_sub.values,
       color='#2C6FBF', alpha=0.85, edgecolor='none')

ax.set_xlabel('Number of Contacts During Campaign', fontsize=11)
ax.set_ylabel('Subscription Rate (%)', fontsize=11)
ax.set_title('More Calls ≠ More Conversions — Diminishing Returns', fontsize=14, fontweight='bold', pad=15)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xticks(range(1, 16))
plt.tight_layout()
plt.savefig('images/04_contacts_vs_conversion.png', bbox_inches='tight')
plt.show()
print('Chart saved to images/04_contacts_vs_conversion.png')

In [ ]:
# ── Chart 5: Subscription Rate by Education Level ──

edu_sub = df.groupby('education')['subscribed'].mean().sort_values(ascending=True) * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(edu_sub.index, edu_sub.values,
               color='#2C6FBF', alpha=0.85, edgecolor='none', height=0.5)

for bar, val in zip(bars, edu_sub.values):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)

ax.set_xlabel('Subscription Rate (%)', fontsize=11)
ax.set_title('Subscription Rate by Education Level', fontsize=14, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('images/05_subscription_by_education.png', bbox_inches='tight')
plt.show()
print('Chart saved to images/05_subscription_by_education.png')

In [ ]:
# ── Chart 6: Correlation Heatmap ──
# Shows which numeric variables are related to each other and to subscription

numeric_cols = ['age', 'campaign', 'pdays', 'previous',
                'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
                'euribor3m', 'nr.employed', 'subscribed']

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Only show lower triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})

ax.set_title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('images/06_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Chart saved to images/06_correlation_heatmap.png')

---
## Section 4: Feature Engineering <a id='4'></a>

In [ ]:
# ── Prepare features and target variable for modeling ──

X = df_encoded.drop(columns=['subscribed'])
y = df_encoded['subscribed']

# ── Split into training (80%) and test (20%) sets ──
# We train the model on 80% of data, then test it on unseen 20%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Test set:     {X_test.shape[0]:,} rows')
print(f'Features:     {X_train.shape[1]}')

# ── Scale numerical features ──
# Logistic Regression needs scaled features to work correctly
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

---
## Section 5: Model Building & Evaluation <a id='5'></a>

In [ ]:
# ── Train three models and compare performance ──
# We test multiple models to find the best performer for this dataset

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
}

results = {}

for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, proba)
    report = classification_report(y_test, preds, output_dict=True)
    results[name] = {
        'model': model,
        'preds': preds,
        'proba': proba,
        'auc': auc,
        'accuracy': report['accuracy'],
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score']
    }
    print(f'{name:25s}  AUC: {auc:.3f}  Accuracy: {report["accuracy"]:.3f}  F1: {report["1"]["f1-score"]:.3f}')

In [ ]:
# ── Chart 7: ROC Curves — Model Comparison ──
# AUC (Area Under Curve) measures how well the model distinguishes subscribers from non-subscribers
# Higher AUC = better model. Random guessing = 0.5

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2C6FBF', '#E05C2A', '#2CA87F']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{name} (AUC = {res['auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.savefig('images/07_roc_curves.png', bbox_inches='tight')
plt.show()
print('Chart saved to images/07_roc_curves.png')

In [ ]:
# ── Chart 8: Confusion Matrix — Best Model ──
# Shows how many predictions were correct vs incorrect

best_model_name = max(results, key=lambda k: results[k]['auc'])
best = results[best_model_name]

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, best['preds'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Did Not Subscribe', 'Subscribed'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('images/08_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Chart saved to images/08_confusion_matrix.png')

In [ ]:
# ── Chart 9: Feature Importance — What Drives Subscriptions? ──
# Shows which customer attributes most influence the prediction

rf_model = results['Random Forest']['model']
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns)
top_features = feat_imp.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top_features.index, top_features.values,
               color='#2C6FBF', edgecolor='none', height=0.6)

for bar, val in zip(bars, top_features.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance Score', fontsize=11)
ax.set_title('Top 15 Features Driving Subscription Predictions', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('images/09_feature_importance.png', bbox_inches='tight')
plt.show()
print('Chart saved to images/09_feature_importance.png')

---
## Section 6: Key Findings & Business Recommendations <a id='6'></a>

In [ ]:
# ── Final model performance summary ──

print('=' * 55)
print('     FINAL MODEL PERFORMANCE SUMMARY')
print('=' * 55)

summary = pd.DataFrame({
    'Model': list(results.keys()),
    'AUC':       [round(r['auc'], 3)       for r in results.values()],
    'Accuracy':  [round(r['accuracy'], 3)  for r in results.values()],
    'Precision': [round(r['precision'], 3) for r in results.values()],
    'Recall':    [round(r['recall'], 3)    for r in results.values()],
    'F1 Score':  [round(r['f1'], 3)        for r in results.values()]
}).set_index('Model')

print(summary.to_string())
print()
print(f'Best Model: {best_model_name}')
print(f'Best AUC:   {best["auc"]:.3f}')

---

## Key Findings

### 1. Economic indicators are the strongest predictors
Features like `euribor3m` (3-month Euribor rate) and `nr.employed` (number of employees) have the highest feature importance. When economic conditions are favorable, customers are more likely to subscribe — suggesting that campaign timing should align with economic cycles.

### 2. Calling customers more than 3 times shows diminishing returns
Subscription rates drop significantly after the 3rd contact. After the 5th contact, rates are near zero. **Marketing spend on customers who have been contacted 4+ times should be reallocated to fresh leads.**

### 3. Student and retired segments over-index on subscriptions
Despite being smaller segments, students and retirees have substantially higher subscription rates than the average. These segments should receive priority in campaign targeting.

### 4. March, September, October, and December are peak subscription months
Campaigns launched in May — the most common launch month — convert at one of the lowest rates. **Shifting campaign volume toward Q4 and early spring could improve ROI significantly.**

---

## Business Recommendations

| Priority | Recommendation | Expected Impact |
|---|---|---|
| High | Stop contacting customers after 3 failed attempts | Reduce wasted call volume by ~35% |
| High | Prioritize students and retirees in campaign targeting | Improve conversion rate by 8-12% |
| Medium | Shift campaign budget from May to Sep-Dec | Better seasonal alignment |
| Medium | Use model risk scores to tier outreach intensity | Focus rep time on highest-probability leads |
| Low | Monitor Euribor rates as a campaign trigger | Launch campaigns when rates are stable |

---
*Analysis by Darius Nobles | [LinkedIn](https://www.linkedin.com/in/dariusnobles/) | [Portfolio](https://dariusnobles.netlify.app/)*